# Export the completed SFT model to safetensors (CPU)

Choose **CPU**, enable **Internet**, and attach the saved TPU posttraining
output dataset. Keep its `training/` folder intact, including `tokenizer.json`,
`corpus_manifest.json`, and `sft/checkpoints/`. No corpus shards or pretrain
dataset are required. The finished run's final step is
`step-00085663-af2e759ac4dc` under `posttrain-20261001-183833/training/sft/checkpoints/`.

This exports only the completed SFT model parameters, including DSpark weights,
with their original BF16/FP32 dtypes. It reads the effective SFT recipe so the
32K context and compressed-layer YaRN factor 4 survive export. Optimizer files
are never loaded. Every model tensor is checksum-checked and compared bit for
bit after safetensors export. The bundled tokenizer must match the training hash.

The output uses the project's portable layout for its custom CPU runtime.
Save the original training checkpoint too if you want to resume training.


In [ ]:
SOURCE_REF = 'codex/midtrain8k-sft16k'
CHECKPOINT = ''  # blank: locate exactly one SFT checkpoint set under /kaggle/input
OUTPUT = '/kaggle/working/nano-dsv41f-sft32k-safetensors'  # use a fresh directory
HF_REPO_ID = ''  # set account/repository to upload; blank skips Hugging Face upload
HF_PRIVATE = True  # visibility when creating a new dataset repository
HF_SECRET_NAME = 'HF_TOKEN'  # Kaggle Secrets: enable your Hugging Face write token


In [ ]:
import importlib.util
import os
from pathlib import Path
import subprocess
import sys

if any(name in sys.modules for name in ('jax', 'nano_dsv41f')):
    raise RuntimeError('Restart the session before updating the source and dependencies')
os.environ['JAX_PLATFORMS'] = 'cpu'
EXPORT_WORK = Path('/kaggle/temp/nano-dsv41f-export')
if EXPORT_WORK.is_symlink():
    raise RuntimeError('The export temporary directory must not be a symlink')
marker = EXPORT_WORK / '.export-workspace'
if EXPORT_WORK.exists() and (not marker.is_file() or marker.read_text().strip() != 'nano-dsv41f-export-workspace-v1'):
    raise RuntimeError('The export temporary directory belongs to another task')
EXPORT_WORK.mkdir(parents=True, exist_ok=True)
marker.write_text('nano-dsv41f-export-workspace-v1')
os.environ['PIP_CACHE_DIR'] = str(EXPORT_WORK / 'pip-cache')
os.environ['HF_HOME'] = str(EXPORT_WORK / 'hf-cache')
os.environ['XDG_CACHE_HOME'] = str(EXPORT_WORK / 'cache')
REPO = EXPORT_WORK / 'source'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/xiayicheng3-code/nano-dsv4.1f.git', str(REPO)], check=True)
if subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain',
                            '--untracked-files=no'], text=True).strip():
    raise RuntimeError('Preserve tracked edits before updating this checkout')
subprocess.run(['git', '-C', str(REPO), 'fetch', '--depth', '1', 'origin', SOURCE_REF], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
if importlib.util.find_spec('torch') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch',
                    '--index-url', 'https://download.pytorch.org/whl/cpu'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '-r', str(REPO / 'requirements-posttrain-inference.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO), '--no-deps'], check=True)
sys.path.insert(0, str(REPO / 'src'))
sys.path.insert(0, str(REPO / 'scripts'))
print('Source commit:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
import json
from export_posttrain_checkpoint import resolve_checkpoint
from pretrain_checkpoint import read_metadata

selected = resolve_checkpoint(CHECKPOINT or '/kaggle/input')
_, checkpoint_manifest = read_metadata(selected)
state = checkpoint_manifest['metadata']
print('Selected checkpoint:', selected)
print('Stage:', state.get('stage'), 'complete:', state.get('stage_complete'))
print('Global step:', state['completed_steps'])
print('SFT context:', state['recipe']['train']['seq_len'])
print('RoPE:', json.dumps(state['recipe']['model']['attention']['rope'], indent=2))
command = [sys.executable, '-u', str(REPO / 'scripts/export_posttrain_checkpoint.py'),
           '--checkpoint', str(selected), '--output', OUTPUT]
subprocess.run(command, check=True)


In [ ]:
from nano_dsv41f.portable_bundle import verify_portable_bundle
from IPython.display import FileLink, display
report = verify_portable_bundle(OUTPUT)
print('Verified tensors:', report['tensor_count'], 'parameters:', report['parameter_count'])
print('Context length:', report['max_position_embeddings'])
for path in sorted(Path(OUTPUT).iterdir()):
    display(FileLink(str(path)))


## Upload to Hugging Face and clean temporary files

Set `HF_REPO_ID` above to `your-account/your-dataset`, add your Hugging Face
write token as the Kaggle secret `HF_TOKEN`, and enable it for this notebook.
This cell creates a **dataset** repo (private by default) and uploads the
verified export files at its root. It runs automatically when the repo ID
is set; leaving it blank saves only the Kaggle output. The key is passed
directly to the API and is not printed or saved as a login file.

The source checkout and caches live outside `/kaggle/working`. After this
cell, the marked temporary workspace is removed, even if an upload fails.
The exported bundle remains available for Kaggle saving and retrying upload.


In [ ]:
from export_bundle_upload import cleanup_export_workspace, upload_export_bundle

def upload_with_kaggle_secret():
    from kaggle_secrets import UserSecretsClient
    try:
        token = UserSecretsClient().get_secret(HF_SECRET_NAME)
    except Exception:
        raise RuntimeError(f'Enable the Kaggle secret {HF_SECRET_NAME} for Hugging Face upload') from None
    try:
        return upload_export_bundle(OUTPUT, HF_REPO_ID, token, private=HF_PRIVATE)
    finally:
        del token

try:
    if HF_REPO_ID:
        uploaded = upload_with_kaggle_secret()
        print('Hugging Face dataset:', f'https://huggingface.co/datasets/{HF_REPO_ID}')
        print('Upload commit:', uploaded.commit_url)
    else:
        print('Hugging Face upload skipped: HF_REPO_ID is blank')
finally:
    cleanup_export_workspace(OUTPUT, EXPORT_WORK)
    print('Temporary export checkout and caches removed')
    print('Saved bundle:', OUTPUT)


## Save and use

Save this notebook version with its outputs, then save the entire
`nano-dsv41f-sft32k-safetensors` folder as a Kaggle dataset. Keep all files
together, especially `model.safetensors`, `config.json`, `tokenizer.json`,
`training_recipe.json`, and `export_manifest.json`. Attach this exported
dataset to `nano_dsv41f_sft_inference_cpu.ipynb`. GitHub supplies the runtime
code; the weights remain in your chosen dataset. Only the bundle is written
under `/kaggle/working`; Kaggle may also generate its normal notebook/log files.
The optional upload cell publishes the same bundle to your Hugging Face dataset.
